# 01 - MobileNetV2 Classifier

Fine-tunes a pre-trained **MobileNetV2** (ImageNet) for 36-class fruits & vegetables classification.

**Strategy**
- Phase 1: freeze the base model, train the new classification head only.
- Phase 2: unfreeze the top layers of MobileNetV2 and fine-tune end-to-end at a low LR.

**Input pipeline:** images are loaded as `[0, 1]` float32. A `Rescaling(2, -1)` layer
inside the model converts them to `[-1, 1]` - the range MobileNetV2 was pre-trained on.

In [ ]:
%matplotlib inline
import sys
import json
import random
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# GPU memory growth (safe no-op if no GPU)
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'Python     : {sys.version.split()[0]}')
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {[g.name for g in gpus] if gpus else "none - CPU only"}')

In [ ]:
ROOT       = Path('.')
TRAIN_ROOT = ROOT / 'train'
TEST_ROOT  = ROOT / 'test'
PLOTS_DIR  = ROOT / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
SEED         = 42
LR_HEAD      = 1e-3     # Phase 1: head only
LR_FINE      = 1e-5     # Phase 2: fine-tune
EPOCHS_HEAD  = 10
EPOCHS_FINE  = 20
FINE_TUNE_AT = 100      # unfreeze base_model layers from this index onward
DROPOUT      = 0.3
IMG_EXTS     = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

assert TRAIN_ROOT.exists(), 'train/ not found - run from project root'
assert TEST_ROOT.exists(),  'test/  not found - run from project root'

## 1. Data Loading

Functions copied verbatim from `00_preprocessing.ipynb` - same `CLASS_NAMES` order is critical.

In [9]:
def discover_classes(root: Path) -> dict:
    """Flatten fruit/vegetable grouping layer -> {class_name: path}."""
    classes = {}
    for cat_dir in sorted(root.iterdir()):
        if not cat_dir.is_dir():
            continue
        for cls_dir in sorted(cat_dir.iterdir()):
            if cls_dir.is_dir():
                classes[cls_dir.name.lower()] = cls_dir
    return classes

def build_file_list(root: Path, class_to_idx: dict) -> tuple:
    """Return (file_path_strings, int_labels) from the nested fruit/veg structure."""
    paths, labels = [], []
    for cat_dir in sorted(root.iterdir()):
        if not cat_dir.is_dir():
            continue
        for cls_dir in sorted(cat_dir.iterdir()):
            if not cls_dir.is_dir():
                continue
            cls = cls_dir.name.lower()
            if cls not in class_to_idx:
                continue
            idx = class_to_idx[cls]
            for f in cls_dir.iterdir():
                if f.suffix.lower() in IMG_EXTS:
                    paths.append(str(f))
                    labels.append(idx)
    return paths, labels

def make_tf_dataset(file_paths, int_labels, n_classes,
                    img_size=(224, 224), batch_size=32,
                    augment=False, shuffle=True, seed=42):
    """Batched tf.data.Dataset. Outputs float32 images in [0, 1]."""

    def load(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        return img, tf.one_hot(label, n_classes)

    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.15)
        img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
        img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
        img = tf.clip_by_value(img, 0.0, 1.0)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((file_paths, int_labels))
    if shuffle:
        ds = ds.shuffle(len(file_paths), seed=seed)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [10]:
train_classes = discover_classes(TRAIN_ROOT)
CLASS_NAMES   = sorted(train_classes.keys())
N_CLASSES     = len(CLASS_NAMES)
CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}

all_paths, all_labels = build_file_list(TRAIN_ROOT, CLASS_TO_IDX)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.2, stratify=all_labels, random_state=SEED
)
test_paths, test_labels = build_file_list(TEST_ROOT, CLASS_TO_IDX)

train_ds = make_tf_dataset(train_paths, train_labels, N_CLASSES,
                           img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                           augment=True,  shuffle=True,  seed=SEED)
val_ds   = make_tf_dataset(val_paths,   val_labels,   N_CLASSES,
                           img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                           augment=False, shuffle=False)
test_ds  = make_tf_dataset(test_paths,  test_labels,  N_CLASSES,
                           img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                           augment=False, shuffle=False)

print(f'Classes     : {N_CLASSES}')
print(f'Train imgs  : {len(train_paths)}')
print(f'Val   imgs  : {len(val_paths)}')
print(f'Test  imgs  : {len(test_paths)}')

# Smoke-test one batch
for x, y in train_ds.take(1):
    print(f'Batch shape : {x.shape}  labels: {y.shape}')
    print(f'Pixel range : [{x.numpy().min():.3f}, {x.numpy().max():.3f}]')

Classes     : 36
Train imgs  : 2492
Val   imgs  : 623
Test  imgs  : 359
Batch shape : (32, 224, 224, 3)  labels: (32, 36)
Pixel range : [0.000, 1.000]


## 2. Model Architecture

```
Input (224, 224, 3)  [0, 1]
  └- Rescaling(×2, -1)       → [-1, 1]  (MobileNetV2 expected range)
  └- MobileNetV2 base        → (7, 7, 1280)
  └- GlobalAveragePooling2D  → (1280,)
  └- BatchNormalization
  └- Dropout(0.3)
  └- Dense(36, softmax)
```

In [11]:
# Load ImageNet base - no top classifier, frozen initially
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
print(f'Base model layers : {len(base_model.layers)}')

# Build full model
inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3), name='input_image')
x       = tf.keras.layers.Rescaling(scale=2.0, offset=-1.0, name='preprocess')(inputs)
x       = base_model(x, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
x       = tf.keras.layers.BatchNormalization(name='head_bn')(x)
x       = tf.keras.layers.Dropout(DROPOUT, name='head_dropout')(x)
outputs = tf.keras.layers.Dense(N_CLASSES, activation='softmax', name='predictions')(x)

model = tf.keras.Model(inputs, outputs, name='mobilenetv2_36cls')

Base model layers : 154


In [12]:
model.summary(show_trainable=True)

print(f'\nTrainable params     : {sum(tf.size(w).numpy() for w in model.trainable_weights):,}')
print(f'Non-trainable params : {sum(tf.size(w).numpy() for w in model.non_trainable_weights):,}')

Model: "mobilenetv2_36cls"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_image (InputLayer)    │ (None, 224, 224, 3)   │          0 │   -   │
├-----------------------------┼-----------------------┼------------┼-------┤
│ preprocess (Rescaling)      │ (None, 224, 224, 3)   │          0 │   -   │
├-----------------------------┼-----------------------┼------------┼-------┤
│ mobilenetv2_1.00_224        │ (None, 7, 7, 1280)    │  2,257,984 │   N   │
│ (Functional)                │                       │            │       │
├-----------------------------┼-----------------------┼------------┼-------┤
│ gap                         │ (None, 1280)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├-----------------------------┼-----------------------┼------------┼-------┤
│ head_bn                     │ (None, 1280)          │      5,120 │   Y   │
│ (BatchNormalization)        │                       │            │       │
├-----------------------------┼-----------------------┼------------┼-------┤
│ head_dropout (Dropout)      │ (None, 1280)          │          0 │   -   │
├-----------------------------┼-----------------------┼------------┼-------┤
│ predictions (Dense)         │ (None, 36)            │     46,116 │   Y   │
└-----------------------------┴-----------------------┴------------┴-------┘

 Total params: 2,309,220 (8.81 MB)

 Trainable params: 48,676 (190.14 KB)

 Non-trainable params: 2,260,544 (8.62 MB)


Trainable params     : 48,676
Non-trainable params : 2,260,544


## 3. Phase 1 - Feature Extraction

MobileNetV2 base is **frozen**. Only the new head (BN + Dropout + Dense) is trained.
This quickly adapts the top layer to the 36-class problem.

In [13]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ph1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    ),
]

print('=== Phase 1: training head only ===')
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks_ph1,
    verbose=1
)

val_acc_ph1 = max(history1.history['val_accuracy'])
print(f'\nBest val accuracy (Phase 1): {val_acc_ph1:.4f}')

=== Phase 1: training head only ===
Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 61s 729ms/step - accuracy: 0.4522 - loss: 2.1234 - val_accuracy: 0.7047 - val_loss: 1.1271 - learning_rate: 0.0010
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 44s 562ms/step - accuracy: 0.7484 - loss: 0.8049 - val_accuracy: 0.7801 - val_loss: 0.7609 - learning_rate: 0.0010
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 67s 862ms/step - accuracy: 0.8190 - loss: 0.5714 - val_accuracy: 0.7994 - val_loss: 0.6577 - learning_rate: 0.0010
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 68s 872ms/step - accuracy: 0.8652 - loss: 0.4190 - val_accuracy: 0.8042 - val_loss: 0.6396 - learning_rate: 0.0010
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 69s 874ms/step - accuracy: 0.8840 - loss: 0.3347 - val_accuracy: 0.7994 - val_loss: 0.6384 - learning_rate: 0.0010
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 76s 957ms/step - accuracy: 0.9073 - loss: 0.2849 - val_accuracy: 0.8074 - val_loss: 0.6331 - learning_rate: 0.0010
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 80s 939m

## 4. Phase 2 - Fine-tuning

Unfreeze the top layers of MobileNetV2 (from index `FINE_TUNE_AT` = 100 onward, out of ~154 total).
Retrain end-to-end at a much lower learning rate to avoid overwriting ImageNet features.

In [14]:
# Unfreeze the base model, then re-freeze everything before FINE_TUNE_AT
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_count   = sum(1 for l in base_model.layers if l.trainable)
untrainable_count = sum(1 for l in base_model.layers if not l.trainable)
print(f'Base model  - trainable: {trainable_count},  frozen: {untrainable_count}')
print(f'Fine-tuning layers [{FINE_TUNE_AT}:]')
for layer in base_model.layers[FINE_TUNE_AT:]:
    print(f'  {layer.name}')

Base model  - trainable: 54,  frozen: 100
Fine-tuning layers [100:]
  block_11_expand_relu
  block_11_depthwise
  block_11_depthwise_BN
  block_11_depthwise_relu
  block_11_project
  block_11_project_BN
  block_11_add
  block_12_expand
  block_12_expand_BN
  block_12_expand_relu
  block_12_depthwise
  block_12_depthwise_BN
  block_12_depthwise_relu
  block_12_project
  block_12_project_BN
  block_12_add
  block_13_expand
  block_13_expand_BN
  block_13_expand_relu
  block_13_pad
  block_13_depthwise
  block_13_depthwise_BN
  block_13_depthwise_relu
  block_13_project
  block_13_project_BN
  block_14_expand
  block_14_expand_BN
  block_14_expand_relu
  block_14_depthwise
  block_14_depthwise_BN
  block_14_depthwise_relu
  block_14_project
  block_14_project_BN
  block_14_add
  block_15_expand
  block_15_expand_BN
  block_15_expand_relu
  block_15_depthwise
  block_15_depthwise_BN
  block_15_depthwise_relu
  block_15_project
  block_15_project_BN
  block_15_add
  block_16_expand
  block_

In [ ]:
# Must recompile after changing trainable states
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ph2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'mobilenet_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
]

print('=== Phase 2: fine-tuning ===')
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks_ph2,
    verbose=1
)

val_acc_ph2 = max(history2.history['val_accuracy'])
print(f'\nBest val accuracy (Phase 2): {val_acc_ph2:.4f}')

## 5. Training History

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']
p1_end   = len(history1.history['accuracy'])
epochs   = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, train_vals, val_vals, title, ylabel in zip(
    axes,
    [acc,  loss],
    [val_acc, val_loss],
    ['Accuracy', 'Loss'],
    ['accuracy', 'loss']
):
    ax.plot(epochs, train_vals, label='train')
    ax.plot(epochs, val_vals,   label='val')
    ax.axvline(x=p1_end + 0.5, color='grey', linestyle='--', linewidth=1,
               label='Phase 1 → 2')
    ax.set(title=title, xlabel='Epoch', ylabel=ylabel)
    ax.legend()

plt.suptitle('MobileNetV2 - Training History', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'mobilenet_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Test Set Evaluation

In [17]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Test loss     : {test_loss:.4f}')
print(f'Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

Test loss     : 0.1900
Test accuracy : 0.9304  (93.04%)


In [18]:
# Collect all true labels and predictions from test_ds
y_true = np.concatenate(
    [np.argmax(lbl.numpy(), axis=1) for _, lbl in test_ds]
)
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

print('=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

=== Classification Report ===
               precision    recall  f1-score   support

        apple      1.000     0.800     0.889        10
       banana      1.000     0.778     0.875         9
     beetroot      1.000     1.000     1.000        10
  bell pepper      1.000     0.300     0.462        10
      cabbage      1.000     1.000     1.000        10
     capsicum      0.500     1.000     0.667        10
       carrot      1.000     0.900     0.947        10
  cauliflower      1.000     1.000     1.000        10
chilli pepper      0.714     1.000     0.833        10
         corn      0.875     0.700     0.778        10
     cucumber      0.909     1.000     0.952        10
     eggplant      1.000     1.000     1.000        10
       garlic      1.000     1.000     1.000        10
       ginger      1.000     1.000     1.000        10
       grapes      1.000     1.000     1.000        10
     jalepeno      1.000     0.900     0.947        10
         kiwi      1.000     1.000

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.4, ax=ax, annot_kws={'size': 7}
)
ax.set(xlabel='Predicted', ylabel='True',
       title='Confusion Matrix - MobileNetV2')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'mobilenet_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Model

In [ ]:
model.save('mobilenet.keras')
print('Model saved → mobilenet.keras')

# Save class names so 04_comparison.ipynb can load them
with open('class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print('Class names saved → class_names.json')

print('\nTo reload:')
print('  model = tf.keras.models.load_model("mobilenet.keras")')
print('  # model expects float32 images in [0, 1] - Rescaling is built in')

## Summary

| Item | Value |
|------|-------|
| Architecture | MobileNetV2 (ImageNet weights) |
| Input | 224 × 224 × 3, float32 in [0, 1] |
| Preprocessing in model | Rescaling(×2, −1) → [−1, 1] |
| Classes | 36 |
| Head | GAP → BN → Dropout(0.3) → Dense(36, softmax) |
| Phase 1 | Frozen base, Adam(1e-3), up to 10 epochs |
| Phase 2 | Unfreeze layers 100+, Adam(1e-5), up to 20 epochs |
| Augmentation | flip, brightness, contrast, saturation |
| Saved model | `mobilenet.keras` (accepts [0,1] input) |